In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS group3_gp.Testing

In [0]:
%sql
DROP TABLE IF EXISTS group3_gp.testing.for_hire_vehicles;
DROP TABLE IF EXISTS group3_gp.testing.high_volume_fhv;
DROP TABLE IF EXISTS group3_gp.testing.green;
DROP TABLE IF EXISTS group3_gp.testing.yellow;

## JSON File Processing and Delta Table Loading in Databricks

This notebook processes JSON files from a specified folder in Databricks, categorizes them based on their filenames, extracts the year from each filename, and loads the data into Spark DataFrames. The DataFrames are then combined by category and written to Delta tables in the Databricks metastore.

- **Source Folder:** `/Volumes/group3_gp/adlslanding/looped_data`
- **Target Database:** `group3_gp.bronze`

In [0]:


import re
from pyspark.sql import functions as F

folder_path = "/Volumes/group3_gp/adlslanding/looped_data"
files = dbutils.fs.ls(folder_path)

category_map = {
    "yellow": "yellow",                # Files containing 'yellow' are mapped to 'yellow' category
    "green": "green",                  # Files containing 'green' are mapped to 'green' category
    "fhv": "high_volume_fhv",          # Files containing 'fhv' are mapped to 'high_volume_fhv' category
    "for hire": "for_hire_vehicles"    # Files containing 'for hire' are mapped to 'for_hire_vehicles' category
}

category_dfs = {cat: [] for cat in category_map.values()}

for file in files:
    file_name = file.name.lower()
    # Extract year (first 4 consecutive digits)
    year_match = re.search(r'(\d{4})', file_name)
    year = int(year_match.group(1)) if year_match else None

    # Determine category
    category = None
    for key, val in category_map.items():
        if key in file_name:
            category = val
            break
    if not category or not year:
        continue

    df = spark.read.option("multiline", "false").json(file.path)
    df = df.withColumn("Year", F.lit(year))
    category_dfs[category].append(df)

# Write to table group3_gp.bronze.{category}
for category, dfs in category_dfs.items():
    if dfs:
        full_df = dfs[0]
        for df in dfs[1:]:
            full_df = full_df.unionByName(df, allowMissingColumns=True)
        full_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"group3_gp.bronze.{category}")

In [0]:


import re
from pyspark.sql import functions as F

folder_path = "/Volumes/group3_gp/adlslanding/looped_data"
files = dbutils.fs.ls(folder_path)

category_map = {
    "yellow": "yellow",                # Files containing 'yellow' are mapped to 'yellow' category
    "green": "green",                  # Files containing 'green' are mapped to 'green' category
    "fhv": "high_volume_fhv",          # Files containing 'fhv' are mapped to 'high_volume_fhv' category
    "for hire": "for_hire_vehicles"    # Files containing 'for hire' are mapped to 'for_hire_vehicles' category
}

category_dfs = {cat: [] for cat in category_map.values()}

for file in files:
    file_name = file.name.lower()
    # Extract year (first 4 consecutive digits)
    year_match = re.search(r'(\d{4})', file_name)
    year = int(year_match.group(1)) if year_match else None

    # Determine category
    category = None
    for key, val in category_map.items():
        if key in file_name:
            category = val
            break
    if not category or not year:
        continue

    df = spark.read.option("multiline", "false").json(file.path)
    df = df.withColumn("Year", F.lit(year))
    category_dfs[category].append(df)

# Write to table group3_gp.bronze.{category}
for category, dfs in category_dfs.items():
    if dfs:
        full_df = dfs[0]
        for df in dfs[1:]:
            full_df = full_df.unionByName(df, allowMissingColumns=True)
        full_df.write.mode("overwrite").saveAsTable(f"group3_gp.bronze.{category}")